In [ ]:
import os
import subprocess
import sys

# ========== CONFIGURATION ==========
REPO_URL = "https://github.com/MuhammadQaiser1921/swin-model.git"
REPO_NAME = "swin-model"
REPO_BRANCH = "main"
REPO_PATH = f"/kaggle/working/{REPO_NAME}"

# ========== CLONE OR PULL REPO ==========
if not os.path.exists(REPO_PATH):
    print(f"📌 Cloning branch '{REPO_BRANCH}' from {REPO_URL}...")
    subprocess.run(["git", "clone", "-b", REPO_BRANCH, REPO_URL], check=True)
else:
    print(f"📌 Repository exists. Fetching updates for branch '{REPO_BRANCH}'...")
    os.chdir(REPO_PATH)
    subprocess.run(["git", "reset", "--hard"], check=True)
    subprocess.run(["git", "fetch", "--all"], check=True)
    subprocess.run(["git", "checkout", REPO_BRANCH], check=True)
    subprocess.run(["git", "pull", "origin", REPO_BRANCH], check=True)
    os.chdir("/kaggle/working")

# ========== SETUP PATHS & REQUIREMENTS ==========
sys.path.append(f"{REPO_PATH}/src")

req_file = f"{REPO_PATH}/requirements.txt"
if os.path.exists(req_file):
    print("Installing requirements...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", req_file], check=True)

print(f"✅ Repository (Branch: {REPO_BRANCH}) is ready and paths are configured.")

In [ ]:
import tensorflow as tf

physical_devices = tf.config.list_physical_devices('GPU')
if physical_devices:
    print("✅ GPU detected")
    for gpu in physical_devices:
        tf.config.experimental.set_memory_growth(gpu, True)

print("TF Version:", tf.__version__)

In [ ]:
from train_audio import load_and_prepare_data

data = load_and_prepare_data()

print("\n📊 Audio Data Preparation Complete:")
print(f"   Training samples: {len(data['train_paths'])}")
print(f"   Validation samples: {len(data['val_paths'])}")
print(f"   Test samples: {len(data['test_paths'])}")

In [ ]:
import importlib
import swin_transformer
import train_audio

importlib.reload(swin_transformer)
importlib.reload(train_audio)

from train_audio import run_training_session, Config

model, history, test_metrics = run_training_session(
    data,
    epochs=Config.epochs,
    batch_size=Config.batch_size,
    lr=Config.lr
)

print("\n✅ Audio training session completed with the latest repository code.")
print("Test metrics:", test_metrics)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train')
plt.plot(history.history['val_accuracy'], label='Val')
plt.title('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train')
plt.plot(history.history['val_loss'], label='Val')
plt.title('Loss')
plt.legend()
plt.show()